In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

PROJECT_PATH = "/content/drive/MyDrive/AI_Resume_Analyzer_and_Job_Recommendation_System"

os.chdir(PROJECT_PATH)

print(os.getcwd())

/content/drive/.shortcut-targets-by-id/16N9xqXDRfgDbBpP92jzc4yE1m_O2Nwfi/AI_Resume_Analyzer_and_Job_Recommendation_System


In [4]:
!pip install -q transformers datasets accelerate evaluate torch scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.3 MB/s eta 0:00:00


In [5]:
from src.config import (
    RESUME_MODEL_VERSION,
    TRAIN_CSV,
    TEST_CSV,
    MODEL_DIR
)

print("=" * 50)
print("Resume Model Version :", RESUME_MODEL_VERSION)
print("Train CSV            :", TRAIN_CSV)
print("Test CSV             :", TEST_CSV)
print("Model Directory      :", MODEL_DIR)
print("=" * 50)

Resume Model Version : resume_v2
Train CSV            : /content/drive/.shortcut-targets-by-id/16N9xqXDRfgDbBpP92jzc4yE1m_O2Nwfi/AI_Resume_Analyzer_and_Job_Recommendation_System/data/processed/train_resume_2.csv
Test CSV             : /content/drive/.shortcut-targets-by-id/16N9xqXDRfgDbBpP92jzc4yE1m_O2Nwfi/AI_Resume_Analyzer_and_Job_Recommendation_System/data/processed/test_resume_2.csv
Model Directory      : /content/drive/.shortcut-targets-by-id/16N9xqXDRfgDbBpP92jzc4yE1m_O2Nwfi/AI_Resume_Analyzer_and_Job_Recommendation_System/saved_models/resume_v2


In [6]:
import os
from src.config import MODEL_DIR

os.makedirs(
    str(MODEL_DIR / "distilbert"),
    exist_ok=True
)

print("Directory created successfully.")

Directory created successfully.


In [7]:
import pandas as pd
from src.config import TRAIN_CSV, TEST_CSV

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

print("Train Shape:", train_df.shape)
print("Test Shape :", test_df.shape)

train_df.head()

Train Shape: (8000, 2)
Test Shape : (2000, 2)


,text,label
0,Education: Web Design Certification Experience...,8
1,Education: Bachelor's in Marketing Experience:...,25
2,Education: MBA Experience: 10 years Skills: Te...,25
3,Education: Master's in Healthcare Administrati...,17
4,Education: Bachelor's in Computer Science Expe...,9


In [8]:
!python -m src.models.distilbert.train

Loading processed datasets...
Train Shape: (8000, 2)
Test Shape : (2000, 2)
Loading tokenizer...
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 257kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 1.49MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 4.43MB/s]
Loading DistilBERT model...
config.json: 100% 483/483 [00:00<00:00, 3.12MB/s]
model.safetensors: 100% 268M/268M [00:01<00:00, 148MB/s] 
Loading weights: 100% 100/100 [00:00<00:00, 7921.40it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- U

In [9]:
from src.config import MODEL_DIR

print("Saved files:")

!ls "{MODEL_DIR}/distilbert"

Saved files:
checkpoint-1000  checkpoint-2500  model.safetensors	 training_args.bin
checkpoint-1500  checkpoint-500   tokenizer_config.json
checkpoint-2000  config.json	  tokenizer.json


In [10]:
!python -m src.models.distilbert.evaluate

Loading weights: 100% 104/104 [00:00<00:00, 2981.92it/s]
100% 250/250 [00:03<00:00, 74.85it/s]

===== DISTILBERT RESULTS =====
Accuracy : 0.9855
Precision: 0.9863
Recall   : 0.9855
F1 Score : 0.9856

Confusion Matrix
[[26  0  0 ...  0  0  0]
 [ 0 23  0 ...  0  0  0]
 [ 0  0 27 ...  0  0  0]
 ...
 [ 0  0  0 ... 31  0  0]
 [ 0  0  0 ...  1 23  0]
 [ 0  0  0 ...  0  0 13]]


In [11]:
from src.models.distilbert.predict import ResumePredictor

predictor = ResumePredictor()

resume = """
Experienced Machine Learning Engineer with Python,
TensorFlow, SQL, Deep Learning,
Computer Vision and NLP.
"""

prediction = predictor.predict(resume)

print(prediction)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

{'category': 'Technology', 'confidence': 0.9983}


In [12]:
import os
from src.config import MODEL_DIR

model_path = MODEL_DIR / "distilbert"

size = 0

for path, dirs, files in os.walk(model_path):
    for f in files:
        fp = os.path.join(path, f)
        size += os.path.getsize(fp)

print(f"Model Size: {size/(1024*1024):.2f} MB")

Model Size: 4089.81 MB


In [13]:
import pandas as pd
from src.config import TRAIN_CSV, TEST_CSV

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

# Normalize text
train_text = (
    train_df["text"]
    .astype(str)
    .str.strip()
    .str.lower()
)

test_text = (
    test_df["text"]
    .astype(str)
    .str.strip()
    .str.lower()
)

duplicates = set(train_text).intersection(set(test_text))

print("=" * 50)
print("Train Samples :", len(train_df))
print("Test Samples  :", len(test_df))
print("Duplicate Resumes :", len(duplicates))
print("=" * 50)

Train Samples : 8000
Test Samples  : 2000
Duplicate Resumes : 0


In [14]:
import pandas as pd
from src.config import TRAIN_CSV

train_df = pd.read_csv(TRAIN_CSV)

print("Number of Classes:", train_df["label"].nunique())

print("\nSamples per Class:")
print(train_df["label"].value_counts().sort_index())

Number of Classes: 42

Samples per Class:
label
0      106
1       93
2      118
3       81
4      136
5       78
6      127
7      107
8      267
9      454
10     146
11      73
12     348
13     155
14      47
15     202
16      72
17     390
18     147
19      86
20     210
21      78
22      54
23     129
24     196
25     370
26     102
27      78
28      79
29     213
30     129
31      69
32     180
33      94
34     112
35     223
36      60
37     110
38    2009
39     122
40      98
41      52
Name: count, dtype: int64
